# Modelo de estimación electoral · Generales 29N
Encuestas (Wikipedia) → swing proporcional por provincia → D'Hondt → Montecarlo → `docs/index.html`.
Los datos de 2023 salen de `data/*.csv` (exportados una vez desde `elecciones.duckdb`).

In [2]:
import pandas as pd, numpy as np

# Datos de 2023 por provincia (exportados de infoelectoral)
prov  = pd.read_csv("data/prov.csv")     # prov, nombre, escanos, blancos
votos = pd.read_csv("data/votos.csv")    # prov, cod_cand, votos, escanos_real
cand  = pd.read_csv("data/cand.csv", keep_default_na=False)  # cod_cand, siglas (sin que "NA" se lea como vacío)
print(len(prov), "provincias |", len(votos), "listas |", len(cand), "candidaturas")

52 provincias | 550 listas | 84 candidaturas


## D'Hondt y validación con 2023

In [3]:
def dhondt(votos_listas: dict, n_escanos: int, validos: int, umbral=0.03):
    elegibles = {c: v for c, v in votos_listas.items() if v >= umbral * validos}
    cocientes = sorted(
        ((v / k, c) for c, v in elegibles.items() for k in range(1, n_escanos + 1)),
        reverse=True,
    )
    res = {c: 0 for c in votos_listas}
    for _, c in cocientes[:n_escanos]:
        res[c] += 1
    return res

def simular(votos_df, prov_df):
    salida = []
    for _, p in prov_df.iterrows():
        sub = votos_df[votos_df.prov == p.prov]
        validos = sub.votos.sum() + p.blancos
        res = dhondt(dict(zip(sub.cod_cand, sub.votos)), int(p.escanos), validos)
        salida += [(p.prov, c, n) for c, n in res.items()]
    return pd.DataFrame(salida, columns=["prov", "cod_cand", "escanos_sim"])

sim = simular(votos, prov)
comp = sim.merge(votos[["prov", "cod_cand", "escanos_real"]], on=["prov", "cod_cand"])
dif = comp[comp.escanos_sim != comp.escanos_real]
print(sim.escanos_sim.sum(), "escaños |", len(dif), "diferencias con 2023")
assert sim.escanos_sim.sum() == 350 and len(dif) == 0, "El D'Hondt no reproduce 2023"

350 escaños | 0 diferencias con 2023


## Cuotas de 2023 y swing proporcional

In [4]:
v = votos.merge(cand, on="cod_cand")          # añade 'siglas' (cabecera nacional)
tot_part = v.groupby("siglas").votos.sum()
cuota23 = (tot_part / tot_part.sum()).sort_values(ascending=False)

def aplicar_swing(votos_df, objetivo):
    """objetivo: dict {siglas: cuota nacional sobre votos a candidaturas}.
    Los partidos no incluidos comparten el resto de forma proporcional."""
    v = votos_df.merge(cand, on="cod_cand").copy()
    tot = v.groupby("siglas").votos.sum()
    total = tot.sum()

    resto = 1 - sum(objetivo.values())
    otros = [s for s in tot.index if s not in objetivo]
    base_otros = tot[otros].sum()

    factor = {}
    for s in tot.index:
        if s in objetivo:
            factor[s] = objetivo[s] * total / tot[s]
        else:
            factor[s] = resto * total / base_otros

    v["votos"] = (v.votos * v.siglas.map(factor)).round().astype(int)
    return v[["prov", "cod_cand", "votos"]]

def resumen(sim_df):
    return (sim_df.merge(cand, on="cod_cand")
                  .groupby("siglas").escanos_sim.sum()
                  .loc[lambda s: s > 0])

base = resumen(sim)   # escaños de 2023 por partido

# Prueba de identidad: con las cuotas de 2023 el resultado debe ser el de 2023
obj_test = {s: cuota23[s] for s in ["PP", "PSOE", "VOX", "SUMAR"]}
t = simular(aplicar_swing(votos, obj_test), prov).merge(votos[["prov", "cod_cand", "escanos_real"]], on=["prov", "cod_cand"])
assert (t.escanos_sim != t.escanos_real).sum() == 0, "El swing nulo no reproduce 2023"
print("Swing nulo OK")

Swing nulo OK


## Encuestas de Wikipedia

In [5]:
import requests, io, re
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Opinion_polling_for_the_next_Spanish_general_election"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (modelo-29n; estimacion electoral)"}, timeout=60)
r.raise_for_status()
html = r.text

# Mismo filtro en ambos lados para que los índices coincidan
tablas = pd.read_html(io.StringIO(html), match="Polling firm")
soup = BeautifulSoup(html, "lxml")
tb_html = [t for t in soup.find_all("table") if "Polling firm" in t.get_text()]
assert len(tablas) == len(tb_html), (len(tablas), len(tb_html))
print(len(tablas), "tablas de encuestas")

def nombre(th):
    txt = th.get_text(strip=True)
    if txt:
        return txt
    img = th.find("img")
    if img and img.get("alt"):
        return img["alt"]
    a = th.find("a")
    return a.get("title", "") if a else ""

12 tablas de encuestas


In [6]:
def quitar_notas(x):
    return re.sub(r"\[.*?\]", "", str(x)).strip()

def pct(x):
    m = re.match(r"(\d+\.\d)", quitar_notas(x))
    return float(m.group(1)) if m else None

def limpiar(i):
    df = tablas[i].copy()
    nombres = [nombre(th) for th in tb_html[i].find_all("tr")[0].find_all("th")]
    assert len(nombres) == df.shape[1], (i, len(nombres), df.shape[1])
    df.columns = nombres

    # filas separadoras de año: todas las celdas iguales y con 4 dígitos
    es_anio = df.apply(lambda r: r.astype(str).nunique() == 1 and re.fullmatch(r"\d{4}", str(r.iloc[0])) is not None, axis=1)
    df["anio"] = df.iloc[:, 0].where(es_anio).astype(float).ffill()
    df = df[~es_anio]

    # solo encuestas reales: muestra numérica o "?"
    ok = df["Sample size"].astype(str).str.fullmatch(r"[\d,]+|\?")
    df = df[ok].copy()
    df["casa"] = df["Polling firm/Commissioner"].map(quitar_notas)
    return df

def cargar(i):
    df = limpiar(i).reset_index(drop=True)
    df["fila"] = df.index
    fijas = ["Polling firm/Commissioner", "Fieldwork date", "Sample size",
             "Turnout", "Lead", "anio", "casa", "fila"]
    partidos = [c for c in df.columns if c not in fijas and c]
    largo = df.melt(id_vars=["fila", "casa", "Fieldwork date", "Sample size"],
                    value_vars=partidos, var_name="partido", value_name="celda")
    largo["pct"] = largo["celda"].map(pct)
    largo["tabla"] = i
    return largo.dropna(subset=["pct"])

enc = pd.concat([cargar(0), cargar(1)], ignore_index=True)

# --- metadatos por encuesta: fecha (año inferido), muestra, casa ---
meses = {m: i for i, m in enumerate(
    ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"], 1)}

meta = (enc[["tabla","fila","casa","Fieldwork date","Sample size"]]
        .drop_duplicates(["tabla","fila"]).sort_values(["tabla","fila"])
        .reset_index(drop=True))

def fecha_fin(txt):
    m = re.findall(r"(\d{1,2})\s+([A-Z][a-z]{2})", quitar_notas(txt))
    return (int(m[-1][0]), meses.get(m[-1][1])) if m else (None, None)

anio, mes_prev, fechas = 2026, 12, []
for t in meta["Fieldwork date"]:
    d_, m_ = fecha_fin(t)
    if m_ is None:
        fechas.append(pd.NaT); continue
    if m_ > mes_prev:
        anio -= 1
    mes_prev = m_
    fechas.append(pd.Timestamp(anio, m_, d_))
meta["fecha"] = fechas
meta["muestra"] = pd.to_numeric(meta["Sample size"].astype(str).str.replace(",", ""), errors="coerce")
meta["pollster"] = meta["casa"].str.split("/").str[0].str.strip()

# --- una fila por encuesta con las 4 claves del modelo ---
p = enc.pivot_table(index=["tabla","fila"], columns="partido", values="pct", aggfunc="first")
p = p.reindex(columns=["PP","PSOE","Vox","FA","Sumar","Podemos"])
p["SUMAR"] = p["FA"].fillna(p["Sumar"]) + p["Podemos"]
p = p.rename(columns={"Vox": "VOX"})[["PP","PSOE","VOX","SUMAR"]].dropna()

d = meta.set_index(["tabla","fila"]).join(p, how="inner").reset_index()
d = d[~d.pollster.isin(["PSOE","PP","Vox","Sumar","Podemos"])].copy()
# Una voz por casa: el CIS y sus recálculos cuentan una vez
d["pollster"] = d.pollster.str.replace(r"\s*\(.*\)", "", regex=True).str.strip()
print(len(meta), "encuestas |", len(d), "con PP, PSOE, Vox y Sumar+Podemos")

166 encuestas | 163 con PP, PSOE, Vox y Sumar+Podemos


## Agregador y escenario central

In [7]:
grandes = ["PP", "PSOE", "VOX", "SUMAR"]
ult = d.fecha.max()

# última encuesta de cada casa en 3 semanas (promedio si hay varias el mismo día)
rec = d[d.fecha >= ult - pd.Timedelta(days=21)].copy()
rec = rec[rec.fecha == rec.groupby("pollster").fecha.transform("max")]
rec = (rec.groupby("pollster", as_index=False)
          .agg(fecha=("fecha", "max"), muestra=("muestra", "mean"),
               PP=("PP", "mean"), PSOE=("PSOE", "mean"),
               VOX=("VOX", "mean"), SUMAR=("SUMAR", "mean")))
rec["dias"] = (ult - rec.fecha).dt.days
rec["w"] = np.sqrt(rec.muestra.fillna(rec.muestra.median())) * 0.5 ** (rec.dias / 14)

polls = {s: float((rec[s] * rec.w).sum() / rec.w.sum()) for s in grandes}
print(len(rec), "casas |", {k: round(v, 1) for k, v in polls.items()})

resto23 = 1 - sum(cuota23[s] for s in grandes)
objetivo = {s: polls[s] / sum(polls.values()) * (1 - resto23) for s in grandes}

esc_central = resumen(simular(aplicar_swing(votos, objetivo), prov))
pd.DataFrame({"2023": base, "central": esc_central}).fillna(0).astype(int).sort_values("central", ascending=False)

14 casas | {'PP': 31.1, 'PSOE': 27.0, 'VOX': 19.0, 'SUMAR': 8.9}


,2023,central
siglas,,
PP,137,127
PSOE,121,108
VOX,33,67
SUMAR,31,18
EH Bildu,6,7
ERC,7,7
JxCAT - JUNTS,7,7
EAJ-PNV,5,6
B.N.G.,1,1


## Montecarlo

In [8]:
v = votos.merge(cand, on="cod_cand").reset_index(drop=True)
names, sig_idx = np.unique(v.siglas.values, return_inverse=True)
base_v = v.votos.values.astype(float)
total = base_v.sum()
tot_s = v.groupby("siglas").votos.sum().reindex(names).values
es_grande = np.isin(names, grandes)
base_otros = tot_s[~es_grande].sum()
esc_prov = prov.set_index("prov").escanos.astype(int).to_dict()
blanc = prov.set_index("prov").blancos.to_dict()
grupos = {p_: np.where(v.prov.values == p_)[0] for p_ in prov.prov}
rng = np.random.default_rng(42)

def dhondt_np(vot, n, validos):
    vv = np.where(vot >= 0.03 * validos, vot, 0.0)
    q = (vv[:, None] / np.arange(1, n + 1)).ravel()
    idx = np.argpartition(q, -n)[-n:]
    return np.bincount(idx // n, minlength=len(vot))

def sim_una(obj, sd_prov):
    objv = np.array([obj.get(s, 0.0) for s in names])
    fac = np.where(es_grande, objv * total / tot_s,
                   (1 - sum(obj.values())) * total / base_otros)
    vot = base_v * fac[sig_idx] * np.exp(rng.normal(0, sd_prov, len(base_v)))
    esc = np.zeros(len(base_v), dtype=int)
    for p_, ix in grupos.items():
        esc[ix] = dhondt_np(vot[ix], esc_prov[p_], vot[ix].sum() + blanc[p_])
    return np.bincount(sig_idx, weights=esc, minlength=len(names))

# Comprobación: sin ruido debe dar el escenario central
chk = pd.Series(sim_una(objetivo, 0.0), index=names)
assert (chk[esc_central.index].astype(int) == esc_central).all(), "Sin ruido no coincide con el central"

N, SD_NAC, SD_PROV = 2000, 0.07, 0.08
base_obj = pd.Series(objetivo)
res = np.zeros((N, len(names)))
for k in range(N):
    sh = base_obj * np.exp(rng.normal(0, SD_NAC, len(base_obj)))
    sh = sh / sh.sum() * base_obj.sum()
    res[k] = sim_una(sh.to_dict(), SD_PROV)
R = pd.DataFrame(res, columns=names)

tabla = pd.DataFrame({"media": R.mean(), "p5": R.quantile(.05),
                      "mediana": R.median(), "p95": R.quantile(.95)}).round(0).astype(int)
tabla = tabla[tabla.p95 > 0].sort_values("media", ascending=False)

bloque_a = ["PP", "VOX", "U.P.N."]
bloque_b = ["PSOE", "SUMAR", "ERC", "JxCAT - JUNTS", "EH Bildu", "EAJ-PNV", "B.N.G.", "CCa"]
print("P(PP+VOX+UPN >= 176):", (R[bloque_a].sum(axis=1) >= 176).mean())
print("P(PSOE+SUMAR+socios >= 176):", (R[bloque_b].sum(axis=1) >= 176).mean())
print("P(PP más escaños que PSOE):", (R["PP"] > R["PSOE"]).mean())
tabla

P(PP+VOX+UPN >= 176): 0.996
P(PSOE+SUMAR+socios >= 176): 0.0025
P(PP más escaños que PSOE): 0.914


,media,p5,mediana,p95
PP,128,115,127,142
PSOE,108,97,108,121
VOX,67,57,67,77
SUMAR,18,13,17,23
EH Bildu,7,6,7,7
ERC,7,7,7,8
JxCAT - JUNTS,7,5,7,8
EAJ-PNV,6,5,6,6
B.N.G.,1,1,1,2
CCa,1,1,1,2


## Dashboard (`docs/index.html`) y CSV de salida

In [9]:
import json, os, datetime
os.makedirs("docs", exist_ok=True)

p_a = float((R[bloque_a].sum(axis=1) >= 176).mean())
p_b = float((R[bloque_b].sum(axis=1) >= 176).mean())
p_pp = float((R["PP"] > R["PSOE"]).mean())

tab = tabla.join(base.rename("2023"), how="left").fillna(0).astype(int)

# Distribución de escaños de cada bloque (para el histograma)
sum_a = R[bloque_a].sum(axis=1).values
sum_b = R[bloque_b].sum(axis=1).values
lo = int(np.floor(min(sum_a.min(), sum_b.min()) / 4) * 4) - 4
hi = int(np.ceil(max(sum_a.max(), sum_b.max()) / 4) * 4) + 4
edges = np.arange(lo, hi + 1, 4)
h_a, _ = np.histogram(sum_a, bins=edges)
h_b, _ = np.histogram(sum_b, bins=edges)

# Encuestas de los últimos 90 días (para el gráfico de evolución)
desde = ult - pd.Timedelta(days=90)
serie = d[d.fecha >= desde].sort_values("fecha")

datos = {
    "generado": datetime.date.today().isoformat(),
    "ultima_encuesta": str(ult.date()),
    "desde": str(desde.date()),
    "n_sim": int(N), "sd_nac": float(SD_NAC), "sd_prov": float(SD_PROV),
    "medias": {k: round(float(v), 1) for k, v in polls.items()},
    "central": {k: int(v) for k, v in esc_central.items() if v > 0},
    "escanos": [{"partido": i, "e2023": int(r["2023"]), "media": int(r.media),
                 "p5": int(r.p5), "p95": int(r.p95)} for i, r in tab.iterrows()],
    "prob": {"pp_vox": p_a, "izq": p_b, "pp_gt_psoe": p_pp},
    "hist": {"edges": [int(x) for x in edges],
             "A": [int(x) for x in h_a], "B": [int(x) for x in h_b],
             "mediaA": float(sum_a.mean()), "mediaB": float(sum_b.mean())},
    "serie": [{"f": r.fecha.strftime("%Y-%m-%d"), "c": r.pollster,
               "PP": float(r.PP), "PSOE": float(r.PSOE),
               "VOX": float(r.VOX), "SUMAR": float(r.SUMAR)} for r in serie.itertuples()],
    "casas": [{"pollster": r.pollster, "fecha": str(r.fecha.date()),
               "PP": round(float(r.PP), 1), "PSOE": round(float(r.PSOE), 1),
               "VOX": round(float(r.VOX), 1), "SUMAR": round(float(r.SUMAR), 1)}
              for r in rec.sort_values("fecha", ascending=False).itertuples()],
}

TPL = r"""<!doctype html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Estimación 29N</title>
<style>
:root{--bg:#fff;--fg:#1a1a1a;--mut:#666;--line:#e3e3e3;--card:#f4f4f4}
@media(prefers-color-scheme:dark){:root{--bg:#121212;--fg:#eee;--mut:#9a9a9a;--line:#2e2e2e;--card:#1e1e1e}}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--fg);font:16px/1.5 system-ui,-apple-system,sans-serif}
main{max-width:900px;margin:0 auto;padding:24px 16px 56px}
h1{font-size:1.7rem;margin:0 0 4px}
h2{font-size:1.1rem;margin:0 0 2px}
.mut{color:var(--mut);font-size:.88rem}
.lead{font-size:1.05rem;margin:12px 0 0}
.panel{background:var(--card);border-radius:14px;padding:16px;margin-top:16px}
svg{width:100%;height:auto;display:block}
.legend{display:flex;flex-wrap:wrap;gap:6px 16px;justify-content:center;margin-top:8px;font-size:.88rem}
.chip i{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:5px}
.rings{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:12px;margin-top:16px}
.ring{background:var(--card);border-radius:14px;padding:14px;text-align:center}
.ring svg{width:110px;margin:0 auto 4px}
.ring span{display:block;font-size:.9rem}
.ax{font-size:11px;fill:var(--mut)}
.big{font-size:46px;font-weight:700;fill:var(--fg)}
.rt{font-size:17px;font-weight:700;fill:var(--fg)}
.lab{font-size:12px;font-weight:600}
.row{display:grid;grid-template-columns:110px 1fr 120px;gap:8px;align-items:center;padding:3px 0;font-size:.9rem}
.track{position:relative;height:16px;background:var(--bg);border-radius:4px}
.bar{position:absolute;top:0;left:0;height:100%;border-radius:4px;opacity:.85}
.wh{position:absolute;top:7px;height:2px;background:var(--fg)}
.num{text-align:right;font-variant-numeric:tabular-nums}
table{width:100%;border-collapse:collapse;font-size:.86rem}
th,td{padding:5px 6px;border-bottom:1px solid var(--line);text-align:right}
th:first-child,td:first-child{text-align:left}
details{margin-top:16px}summary{cursor:pointer;font-weight:600}
ul{padding-left:20px;margin:8px 0 0}li{margin-bottom:6px}
</style></head><body><main>
<h1>Estimación de escaños · Generales 29N</h1>
<div class="mut" id="sub"></div>
<p class="lead" id="lead"></p>

<div class="panel">
<h2>Escenario central</h2>
<div class="mut">350 escaños del Congreso con la media actual de encuestas (cada punto es un escaño)</div>
<div id="hemi"></div><div class="legend" id="leg"></div>
</div>

<div class="rings" id="rings"></div>

<div class="panel">
<h2>¿Cuánto varía el resultado?</h2>
<div class="mut">Escaños del bloque en cada una de las simulaciones. La línea marca la mayoría absoluta (176).</div>
<div id="hist"></div><div class="legend" id="hleg"></div>
</div>

<div class="panel">
<h2>Evolución de las encuestas</h2>
<div class="mut">Cada punto es una encuesta; la línea es la media móvil de 15 días (simple, no es la media ponderada del modelo).</div>
<div id="polls"></div><div class="legend" id="pleg"></div>
</div>

<div class="panel">
<h2>Rango de escaños por partido</h2>
<div class="mut">Barra: media de las simulaciones. Línea negra: rango del 90 %.</div>
<div id="bars" style="margin-top:8px"></div>
</div>

<details class="panel"><summary>Encuestas usadas en la media</summary>
<div style="overflow-x:auto;margin-top:8px"><table id="tab"></table></div></details>

<details class="panel"><summary>Metodología y límites</summary><ul id="met"></ul></details>
</main><script>
const D = __DATA__;
const $ = id => document.getElementById(id);
const esc = s => String(s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));
const COL = {PP:"#1f77b4",PSOE:"#e03c31",VOX:"#63b32e",SUMAR:"#a259d9",ERC:"#ffb300","JxCAT - JUNTS":"#4dd0e1","EH Bildu":"#aed581","EAJ-PNV":"#00796b","B.N.G.":"#80cbc4",CCa:"#ff8a65","U.P.N.":"#5c6bc0","CUP-PR":"#fdd835","NC-bc":"#8d6e63"};
const NOM = {VOX:"Vox",SUMAR:"Sumar+Podemos","JxCAT - JUNTS":"Junts","EAJ-PNV":"PNV","B.N.G.":"BNG","U.P.N.":"UPN","CUP-PR":"CUP","NC-bc":"Nueva Canarias"};
const colr = p => COL[p] || "#9e9e9e";
const nom = p => NOM[p] || p;
const pct = x => x > 0 && x < 0.001 ? "<0,1 %" : (x * 100).toFixed(1).replace(".", ",") + " %";
const MES = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"];
const fmt = s => { const d = new Date(s + "T00:00:00Z"); return d.getUTCDate() + " " + MES[d.getUTCMonth()]; };

function hemiciclo(central) {
  const orden = ["SUMAR","PSOE","ERC","EH Bildu","B.N.G.","EAJ-PNV","JxCAT - JUNTS","CCa","U.P.N.","PP","VOX"];
  const parts = orden.filter(k => central[k]).concat(Object.keys(central).filter(k => !orden.includes(k) && central[k]));
  const N = parts.reduce((a, k) => a + central[k], 0);
  const rows = 10, r0 = 135, r1 = 325;
  const radios = Array.from({length: rows}, (_, i) => r0 + (r1 - r0) * i / (rows - 1));
  const sr = radios.reduce((a, b) => a + b, 0);
  const cnt = radios.map(r => Math.round(N * r / sr));
  cnt[rows - 1] += N - cnt.reduce((a, b) => a + b, 0);
  const seats = [];
  radios.forEach((r, i) => {
    const n = cnt[i];
    for (let j = 0; j < n; j++) seats.push({ang: n === 1 ? Math.PI / 2 : Math.PI - Math.PI * j / (n - 1), r});
  });
  seats.sort((a, b) => (b.ang - a.ang) || (a.r - b.r));
  let k = 0, out = "";
  parts.forEach(p => {
    for (let c = 0; c < central[p]; c++) {
      const s = seats[k++];
      out += `<circle cx="${(350 + s.r * Math.cos(s.ang)).toFixed(1)}" cy="${(352 - s.r * Math.sin(s.ang)).toFixed(1)}" r="8" fill="${colr(p)}"><title>${esc(nom(p))}</title></circle>`;
    }
  });
  const svg = `<svg viewBox="0 0 700 370" role="img" aria-label="Hemiciclo de escaños">${out}<text x="350" y="326" text-anchor="middle" class="big">${N}</text><text x="350" y="350" text-anchor="middle" class="ax">mayoría absoluta: 176</text></svg>`;
  return {svg, parts};
}

function ring(p, c, t) {
  const C = 2 * Math.PI * 40, d = C * Math.max(0, Math.min(1, p));
  return `<div class="ring"><svg viewBox="0 0 100 100"><circle cx="50" cy="50" r="40" fill="none" stroke="var(--line)" stroke-width="10"/><circle cx="50" cy="50" r="40" fill="none" stroke="${c}" stroke-width="10" stroke-dasharray="${d.toFixed(1)} ${C.toFixed(1)}" transform="rotate(-90 50 50)"/><text x="50" y="56" text-anchor="middle" class="rt">${pct(p)}</text></svg><span>${t}</span></div>`;
}

function histograma(H) {
  const W = 700, Ht = 250, L = 12, Rr = 12, T = 16, B = 32;
  const maxY = Math.max(...H.A, ...H.B, 1);
  const x0 = H.edges[0], x1 = H.edges[H.edges.length - 1];
  const X = v => L + (v - x0) / (x1 - x0) * (W - L - Rr);
  const Y = v => T + (1 - v / maxY) * (Ht - T - B);
  let s = "";
  for (let i = 0; i < H.A.length; i++) {
    const xa = X(H.edges[i]), w = Math.max(X(H.edges[i + 1]) - xa - 1, 1);
    [[H.B, "#e03c31"], [H.A, "#1f77b4"]].forEach(([arr, c]) => {
      if (arr[i] > 0) s += `<rect x="${xa.toFixed(1)}" y="${Y(arr[i]).toFixed(1)}" width="${w.toFixed(1)}" height="${(Ht - B - Y(arr[i])).toFixed(1)}" fill="${c}" opacity=".7"/>`;
    });
  }
  for (let v = Math.ceil(x0 / 20) * 20; v <= x1; v += 20)
    s += `<text x="${X(v).toFixed(1)}" y="${Ht - B + 16}" text-anchor="middle" class="ax">${v}</text>`;
  s += `<line x1="${L}" x2="${W - Rr}" y1="${Ht - B}" y2="${Ht - B}" stroke="var(--line)"/>`;
  const m = X(176);
  s += `<line x1="${m.toFixed(1)}" x2="${m.toFixed(1)}" y1="${T}" y2="${Ht - B}" stroke="var(--fg)" stroke-dasharray="4 3"/><text x="${(m + 6).toFixed(1)}" y="${T + 8}" class="ax">176</text>`;
  return `<svg viewBox="0 0 ${W} ${Ht}" role="img" aria-label="Distribución de escaños por bloque">${s}</svg>`;
}

function grafico(S, desde) {
  const W = 700, Ht = 300, L = 34, Rr = 128, T = 14, B = 28;
  const P = ["PP", "PSOE", "VOX", "SUMAR"];
  const pts = S.map(s => Object.assign({t: Date.parse(s.f + "T00:00:00Z")}, s));
  const t0 = Date.parse(desde + "T00:00:00Z"), t1 = Math.max(...pts.map(p => p.t), t0 + 86400000);
  const vals = pts.flatMap(p => P.map(k => p[k]));
  const ymin = Math.max(0, Math.floor((Math.min(...vals) - 1) / 5) * 5);
  const ymax = Math.ceil((Math.max(...vals) + 1) / 5) * 5;
  const X = t => L + (t - t0) / (t1 - t0) * (W - L - Rr);
  const Y = v => T + (1 - (v - ymin) / (ymax - ymin)) * (Ht - T - B);
  let s = "";
  for (let v = ymin; v <= ymax; v += 5)
    s += `<line x1="${L}" x2="${W - Rr}" y1="${Y(v).toFixed(1)}" y2="${Y(v).toFixed(1)}" stroke="var(--line)"/><text x="${L - 6}" y="${(Y(v) + 4).toFixed(1)}" text-anchor="end" class="ax">${v}</text>`;
  for (let t = t0; t <= t1; t += 30 * 86400000)
    s += `<text x="${X(t).toFixed(1)}" y="${Ht - 8}" text-anchor="middle" class="ax">${fmt(new Date(t).toISOString().slice(0, 10))}</text>`;
  P.forEach(k => {
    pts.forEach(p => { s += `<circle cx="${X(p.t).toFixed(1)}" cy="${Y(p[k]).toFixed(1)}" r="2.6" fill="${colr(k)}" opacity=".3"/>`; });
    let d = "", last = null;
    for (let t = t0; t <= t1; t += 2 * 86400000) {
      const w = pts.filter(p => Math.abs(p.t - t) <= 7.5 * 86400000);
      if (w.length < 2) continue;
      const v = w.reduce((a, p) => a + p[k], 0) / w.length;
      d += (d ? "L" : "M") + X(t).toFixed(1) + " " + Y(v).toFixed(1);
      last = {t, v};
    }
    if (d) s += `<path d="${d}" fill="none" stroke="${colr(k)}" stroke-width="2.5" stroke-linejoin="round"/>`;
    if (last) s += `<text x="${(X(last.t) + 6).toFixed(1)}" y="${(Y(last.v) + 4).toFixed(1)}" class="lab" fill="${colr(k)}">${esc(nom(k))} ${last.v.toFixed(1).replace(".", ",")}</text>`;
  });
  return `<svg viewBox="0 0 ${W} ${Ht}" role="img" aria-label="Evolución de las encuestas">${s}</svg>`;
}

$("sub").textContent = `Actualizado ${D.generado} · última encuesta ${D.ultima_encuesta} · ${D.n_sim} simulaciones · modelo experimental, no es una predicción`;
const lider = Object.entries(D.central).sort((a, b) => b[1] - a[1]).slice(0, 4).map(([p, n]) => `${nom(p)} ${n}`).join(" · ");
$("lead").textContent = "Con la media actual de encuestas, el modelo estima: " + lider + ".";

const H = hemiciclo(D.central);
$("hemi").innerHTML = H.svg;
$("leg").innerHTML = H.parts.map(p => `<span class="chip"><i style="background:${colr(p)}"></i>${esc(nom(p))} <b>${D.central[p]}</b></span>`).join("");

$("rings").innerHTML =
  ring(D.prob.pp_vox, "#1f77b4", "PP + Vox + UPN llegan a 176 escaños") +
  ring(D.prob.izq, "#e03c31", "PSOE y socios llegan a 176 escaños") +
  ring(D.prob.pp_gt_psoe, "#6b6b6b", "El PP queda por delante del PSOE");

$("hist").innerHTML = histograma(D.hist);
$("hleg").innerHTML =
  `<span class="chip"><i style="background:#1f77b4"></i>PP + Vox + UPN (media ${Math.round(D.hist.mediaA)})</span>` +
  `<span class="chip"><i style="background:#e03c31"></i>PSOE + Sumar + ERC + Junts + Bildu + PNV + BNG + CC (media ${Math.round(D.hist.mediaB)})</span>`;

$("polls").innerHTML = grafico(D.serie, D.desde);
$("pleg").innerHTML = ["PP", "PSOE", "VOX", "SUMAR"].map(p => `<span class="chip"><i style="background:${colr(p)}"></i>${esc(nom(p))}</span>`).join("");

const mx = Math.max(...D.escanos.map(e => e.p95), 1);
const w = v => (v / mx * 100) + "%";
$("bars").innerHTML = D.escanos.map(e =>
  `<div class="row"><span>${esc(nom(e.partido))}</span><div class="track"><div class="bar" style="width:${w(e.media)};background:${colr(e.partido)}"></div><div class="wh" style="left:${w(e.p5)};width:${w(e.p95 - e.p5)}"></div></div><span class="num">${e.media} (${e.p5}–${e.p95})</span></div>`).join("");

const hd = ["Casa", "Fecha", "PP", "PSOE", "Vox", "Sumar+Podemos"];
$("tab").innerHTML = "<tr>" + hd.map(x => `<th>${x}</th>`).join("") + "</tr>" +
  D.casas.map(c => `<tr><td>${esc(c.pollster)}</td><td>${c.fecha}</td><td>${c.PP}</td><td>${c.PSOE}</td><td>${c.VOX}</td><td>${c.SUMAR}</td></tr>`).join("") +
  `<tr><td><b>Media ponderada</b></td><td></td><td><b>${D.medias.PP}</b></td><td><b>${D.medias.PSOE}</b></td><td><b>${D.medias.VOX}</b></td><td><b>${D.medias.SUMAR}</b></td></tr>`;

$("met").innerHTML = [
  "Datos: resultados de 2023 por provincia (Ministerio del Interior, infoelectoral) y encuestas recopiladas de Wikipedia.",
  "Agregador: última encuesta de cada casa en las 3 últimas semanas, ponderada por la raíz de la muestra y con un peso que se reduce a la mitad cada 14 días. Cada casa cuenta una vez (el CIS y sus recálculos, también).",
  "Swing proporcional: los votos de cada partido en cada provincia se reescalan por el cociente entre su cuota actual y la de 2023. El resto de partidos se mantiene en su peso de 2023.",
  "Reparto: D'Hondt por provincia con umbral del 3 %. Con los votos de 2023 reproduce los 350 escaños reales.",
  `Montecarlo: ${D.n_sim} simulaciones con ruido nacional (${(D.sd_nac * 100).toFixed(0)} % relativo) y provincial (${(D.sd_prov * 100).toFixed(0)} %). Ambos valores se fijaron a mano y no están calibrados con un backtest, por lo que las probabilidades son orientativas.`,
  "Límites: Sumar y Podemos van fusionados, lo que sobrestima a la izquierda alternativa; las listas nuevas no se modelan; el swing uniforme ignora diferencias regionales; sumar 176 escaños no equivale a formar gobierno."
].map(x => `<li>${esc(x)}</li>`).join("");
</script></body></html>"""

open("docs/index.html", "w", encoding="utf-8").write(
    TPL.replace("__DATA__", json.dumps(datos, ensure_ascii=False)))

21663

In [10]:
import os
os.makedirs("docs", exist_ok=True)
tab.to_csv("docs/escanos.csv")
d[["pollster", "casa", "fecha", "muestra", "PP", "PSOE", "VOX", "SUMAR"]].to_csv("docs/encuestas.csv", index=False)
print(sorted(os.listdir("docs")))

['encuestas.csv', 'escanos.csv', 'index.html']
